In [ ]:
# ============================================================
# BIASCOPE
# Notebook 02 — Dataset Preparation & Exploratory Data Analysis
# ============================================================
#
# Objective:
# 1. Load the media-bias dataset
# 2. Understand the dataset structure
# 3. Inspect labels and class distribution
# 4. Clean noisy / duplicate samples
# 5. Analyze text statistics
# 6. Create reproducible train/validation/test splits
# 7. Save processed datasets for model training
#
# Project:
# Multilingual & Code-Mixed Explainable News Bias Detection
# ============================================================

PROJECT_NAME = "BIASCOPE"
DATASET_NAME = "MBIC / MBIB"
TASK = "Text-Level Media Bias Detection"

print(f"Project : {PROJECT_NAME}")
print(f"Dataset : {DATASET_NAME}")
print(f"Task    : {TASK}")

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split

from datasets import load_dataset, Dataset, DatasetDict

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed:", SEED)

In [ ]:
BASE_DIR = Path("/content/biascope")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

In [ ]:
from datasets import get_dataset_config_names

dataset_name = "mediabiasgroup/mbib-base"

configs = get_dataset_config_names(dataset_name)

print("Available configurations:")
for config in configs:
    print("-", config)

In [ ]:
dataset = load_dataset(
    "mediabiasgroup/mbib-base"
)

print(dataset)

In [ ]:
print("Available splits:")

for split in dataset:
    print(f"- {split}")

In [ ]:
bias_dataset = dataset["text_level_bias"]

print(bias_dataset)

In [ ]:
print(bias_dataset.features)

In [ ]:
print("Number of samples:", len(bias_dataset))

In [ ]:
train_df = bias_dataset.to_pandas()

print("Shape:", train_df.shape)

In [ ]:
print("Columns:")
print(train_df.columns.tolist())

In [ ]:
display(train_df.head(10))

In [ ]:
print("Data types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isnull().sum())

In [ ]:
print("Label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nLabel percentages:")
print(
    train_df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=train_df,
    x="label"
)

plt.title("MBIB Text-Level Bias Label Distribution")
plt.xlabel("Label")
plt.ylabel("Number of Samples")

plt.show()

In [ ]:
for label in sorted(train_df["label"].unique()):
    print(f"\n========== LABEL {label} ==========")

    samples = train_df[train_df["label"] == label]["text"].head(5)

    for i, text in enumerate(samples, 1):
        print(f"\n{i}. {text}")

In [ ]:
print("Duplicate complete rows:", train_df.duplicated().sum())

print(
    "Duplicate texts:",
    train_df["text"].duplicated().sum()
)

In [ ]:
duplicate_texts = train_df[
    train_df["text"].duplicated(keep=False)
].sort_values("text")

display(duplicate_texts.head(20))

In [ ]:
train_df["char_length"] = train_df["text"].str.len()

train_df["word_length"] = train_df["text"].str.split().str.len()

print(train_df[["char_length", "word_length"]].describe())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=train_df,
    x="word_length",
    bins=50
)

plt.title("Distribution of Text Length")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print("Texts with <= 3 words:")

short_texts = train_df[
    train_df["word_length"] <= 3
]

print("Count:", len(short_texts))

display(
    short_texts[["text", "label"]].head(20)
)

In [ ]:
print("Number of unique dataset IDs:")
print(train_df["dataset_id"].nunique())

print("\nDataset ID distribution:")
print(train_df["dataset_id"].value_counts().sort_index())

In [ ]:
dataset_label_table = pd.crosstab(
    train_df["dataset_id"],
    train_df["label"],
    normalize="index"
) * 100

dataset_label_table = dataset_label_table.round(2)

display(dataset_label_table)

In [ ]:
short_distribution = pd.crosstab(
    train_df["word_length"] <= 3,
    train_df["label"]
)

print(short_distribution)

In [ ]:
print(
    train_df.groupby(
        train_df["word_length"] <= 3
    )["label"].value_counts()
)

In [ ]:
short_samples = (
    train_df[
        train_df["word_length"] <= 3
    ][["text", "label", "dataset_id", "word_length"]]
    .sort_values("word_length")
)

display(short_samples.head(50))

In [ ]:
patterns = {
    "url": r"https?://|www\.",
    "reddit": r"\breddit\b",
    "archive": r"\barchive\b",
    "bot": r"\bbot\b",
    "removeddit": r"\bremoveddit\b",
    "snapshot": r"\bsnapshot\b"
}

for name, pattern in patterns.items():
    train_df[f"has_{name}"] = (
        train_df["text"]
        .str.contains(pattern, case=False, regex=True, na=False)
    )

    print(
        f"{name:12s}: "
        f"{train_df[f'has_{name}'].sum():,}"
    )

In [ ]:
suspicious_mask = (
    train_df["has_url"]
    | train_df["has_reddit"]
    | train_df["has_archive"]
    | train_df["has_bot"]
    | train_df["has_removeddit"]
    | train_df["has_snapshot"]
)

print(
    "Potentially noisy/source-artifact samples:",
    suspicious_mask.sum()
)

display(
    train_df.loc[
        suspicious_mask,
        ["text", "label", "dataset_id"]
    ].head(30)
)

In [ ]:
def non_alpha_ratio(text):
    if not isinstance(text, str) or len(text) == 0:
        return 1.0

    non_alpha = sum(
        not ch.isalpha()
        and not ch.isspace()
        for ch in text
    )

    return non_alpha / len(text)


train_df["non_alpha_ratio"] = train_df["text"].apply(
    non_alpha_ratio
)

print(
    train_df["non_alpha_ratio"].describe()
)

In [ ]:
print("Texts with > 500 words:")

long_samples = train_df[
    train_df["word_length"] > 500
]

print("Count:", len(long_samples))

display(
    long_samples[
        ["text", "label", "dataset_id", "word_length"]
    ].sort_values(
        "word_length",
        ascending=False
    ).head(20)
)

In [ ]:
print("========== DATA QUALITY SUMMARY ==========")

print("Total samples:", len(train_df))

print(
    "Missing texts:",
    train_df["text"].isna().sum()
)

print(
    "Duplicate texts:",
    train_df["text"].duplicated().sum()
)

print(
    "Texts <= 3 words:",
    (train_df["word_length"] <= 3).sum()
)

print(
    "Texts > 500 words:",
    (train_df["word_length"] > 500).sum()
)

print(
    "URL-containing texts:",
    train_df["has_url"].sum()
)

print(
    "Reddit-containing texts:",
    train_df["has_reddit"].sum()
)

print(
    "Archive-containing texts:",
    train_df["has_archive"].sum()
)

print(
    "Bot-containing texts:",
    train_df["has_bot"].sum()
)

In [ ]:
print("===== DATASET ID ANALYSIS =====")

for dataset_id in sorted(train_df["dataset_id"].unique()):

    subset = train_df[train_df["dataset_id"] == dataset_id]

    print(f"\nDataset ID: {dataset_id}")
    print(f"Samples: {len(subset):,}")
    print(
        "Label distribution:",
        subset["label"].value_counts().sort_index().to_dict()
    )

In [ ]:
dataset_label_counts = pd.crosstab(
    train_df["dataset_id"],
    train_df["label"]
)

dataset_label_counts.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Label Distribution Across Dataset Sources")
plt.xlabel("Dataset ID")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)

plt.show()

In [ ]:
subset_19 = train_df[
    train_df["dataset_id"] == 19
]

display(
    subset_19[
        ["text", "label", "dataset_id"]
    ].sample(
        min(20, len(subset_19)),
        random_state=42
    )
)

In [ ]:
subset_76 = train_df[
    train_df["dataset_id"] == 76
]

display(
    subset_76[
        ["text", "label", "dataset_id"]
    ].sample(
        20,
        random_state=42
    )
)

In [ ]:
garbage_mask = (
    (train_df["word_length"] == 1) &
    (
        train_df["text"].str.fullmatch(
            r"[A-Za-z]+",
            na=False
        )
    ) &
    (
        train_df["text"].str.len() <= 3
    )
)

print(
    "Potential single-token garbage:",
    garbage_mask.sum()
)

display(
    train_df.loc[
        garbage_mask,
        ["text", "label", "dataset_id"]
    ].head(100)
)

In [ ]:
repeated_char_mask = train_df["text"].str.contains(
    r"(.)\1{4,}",
    regex=True,
    na=False
)

print(
    "Potential repeated-character noise:",
    repeated_char_mask.sum()
)

display(
    train_df.loc[
        repeated_char_mask,
        ["text", "label", "dataset_id"]
    ].head(50)
)

In [ ]:
gibberish_mask = (
    (train_df["word_length"] == 1) &
    (train_df["text"].str.len() >= 10) &
    (train_df["text"].str.fullmatch(
        r"[A-Za-z]+",
        na=False
    ))
)

print(
    "Potential single-token gibberish:",
    gibberish_mask.sum()
)

display(
    train_df.loc[
        gibberish_mask,
        ["text", "label", "dataset_id"]
    ].head(100)
)

In [ ]:
train_df["flag_short"] = train_df["word_length"] <= 3

train_df["flag_repeated_chars"] = repeated_char_mask

train_df["flag_gibberish"] = gibberish_mask

train_df["flag_source_artifact"] = suspicious_mask

train_df["flag_extreme_length"] = train_df["word_length"] > 500

In [ ]:
quality_flags = [
    "flag_short",
    "flag_repeated_chars",
    "flag_gibberish",
    "flag_source_artifact",
    "flag_extreme_length"
]

for col in quality_flags:
    print(
        f"{col:30s}: "
        f"{train_df[col].sum():,}"
    )

In [ ]:
display(
    train_df.loc[
        repeated_char_mask,
        ["text", "label", "dataset_id", "word_length"]
    ].head(100)
)

In [ ]:
display(
    train_df.loc[
        suspicious_mask,
        ["text", "label", "dataset_id", "word_length"]
    ].head(100)
)

In [ ]:
print("DATASET ID 19")

display(
    train_df[
        train_df["dataset_id"] == 19
    ][
        ["text", "label", "word_length"]
    ].head(50)
)

In [ ]:
print("DATASET ID 76")

display(
    train_df[
        train_df["dataset_id"] == 76
    ][
        ["text", "label", "word_length"]
    ].head(50)
)

In [ ]:
# ============================================================
# FINAL MINIMAL CLEANING
# ============================================================

clean_df = train_df.copy()

# Normalize whitespace at the beginning/end
clean_df["text"] = (
    clean_df["text"]
    .astype(str)
    .str.strip()
)

# Remove genuinely empty texts
clean_df = clean_df[
    clean_df["text"].str.len() > 0
].copy()

# Remove exact duplicate texts
clean_df = clean_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

print("Original samples :", len(train_df))
print("Cleaned samples  :", len(clean_df))
print("Removed samples  :", len(train_df) - len(clean_df))

In [ ]:
print("Label distribution after cleaning:")

print(
    clean_df["label"]
    .value_counts()
    .sort_index()
)

print("\nPercentages:")

print(
    clean_df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

In [ ]:
print("Dataset source distribution:")

print(
    clean_df["dataset_id"]
    .value_counts()
    .sort_index()
)

print("\nSource × label:")

display(
    pd.crosstab(
        clean_df["dataset_id"],
        clean_df["label"]
    )
)

In [ ]:
stratify=clean_df["label"]

In [ ]:
clean_df["stratify_key"] = (
    clean_df["dataset_id"].astype(str)
    + "_"
    + clean_df["label"].astype(str)
)

print(
    clean_df["stratify_key"]
    .value_counts()
)

In [ ]:
train_df_final, temp_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    stratify=clean_df["stratify_key"]
)

print("Train:", len(train_df_final))
print("Temp :", len(temp_df))

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["stratify_key"]
)

print("Train:", len(train_df_final))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
def show_distribution(name, df):

    print(f"\n===== {name.upper()} =====")

    print("Samples:", len(df))

    print("\nLabels:")
    print(
        df["label"]
        .value_counts()
        .sort_index()
    )

    print("\nPercent:")
    print(
        df["label"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )


show_distribution("Train", train_df_final)
show_distribution("Validation", val_df)
show_distribution("Test", test_df)

In [ ]:
for name, df in [
    ("Train", train_df_final),
    ("Validation", val_df),
    ("Test", test_df)
]:

    print(f"\n===== {name.upper()} =====")

    distribution = (
        df["dataset_id"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print(distribution)

In [ ]:
train_texts = set(train_df_final["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

In [ ]:
for df in [train_df_final, val_df, test_df]:
    df.drop(
        columns=["stratify_key"],
        inplace=True,
        errors="ignore"
    )

In [ ]:
print("Final columns:")
print(train_df_final.columns.tolist())

In [ ]:
TRAIN_PATH = PROCESSED_DIR / "mbib_train.csv"
VAL_PATH = PROCESSED_DIR / "mbib_validation.csv"
TEST_PATH = PROCESSED_DIR / "mbib_test.csv"

train_df_final.to_csv(
    TRAIN_PATH,
    index=False
)

val_df.to_csv(
    VAL_PATH,
    index=False
)

test_df.to_csv(
    TEST_PATH,
    index=False
)

print("Saved successfully!")

print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

In [ ]:
from pathlib import Path
import os

print("CURRENT WORKING DIRECTORY:")
print(os.getcwd())

print("\nBIASCOPE DIRECTORY EXISTS:")
print(Path("/content/biascope").exists())

print("\nALL CSV FILES UNDER /content:")
csv_files = list(Path("/content").rglob("*.csv"))

if not csv_files:
    print("❌ NO CSV FILES FOUND")
else:
    for f in csv_files:
        print(f)

print("\nPROCESSED DIRECTORY:")
processed = Path("/content/biascope/data/processed")

if processed.exists():
    for f in processed.iterdir():
        print(f)
else:
    print("❌ /content/biascope/data/processed DOES NOT EXIST")

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Permanent storage location
DRIVE_PROCESSED_DIR = Path(
    "/content/drive/MyDrive/BIASCOPE/data/processed"
)

DRIVE_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Copy processed datasets from Colab runtime → Google Drive
files_to_copy = [
    "mbib_train.csv",
    "mbib_validation.csv",
    "mbib_test.csv"
]

for filename in files_to_copy:

    source = Path(
        "/content/biascope/data/processed"
    ) / filename

    destination = DRIVE_PROCESSED_DIR / filename

    shutil.copy2(source, destination)

    print(f"Copied: {filename}")

print("\n✅ All processed datasets backed up to Google Drive.")

print("\nGoogle Drive location:")
print(DRIVE_PROCESSED_DIR)

print("\nFiles:")
for file in DRIVE_PROCESSED_DIR.iterdir():
    print(file)